# 벤포드 법칙 검정

2의 거듭제곱, 미국 카운티 인구, 단위 변환, 균등 난수로 첫자리 분포를 카이제곱과 MAD로 검정합니다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/benford_law_census.ipynb)

Colab에서 CPU 런타임을 선택하고 위에서부터 셀을 실행하세요. 첫 코드 셀에서 `QUICK = True`로 바꾸면 축소 조건으로 실행합니다.

In [ ]:
QUICK = False

import math
import random

DIGITS = range(1, 10)
BENFORD = tuple(math.log10(1 + 1 / digit) for digit in DIGITS)
SEED = 20231119


def leading_digit(numerator, denominator=1):
    """양의 유리수의 첫 유효 숫자를 정수 연산으로 구한다."""
    # log10(2)의 근사값 1233/4096으로 지수를 추정한 뒤 정수 비교로 보정한다.
    exponent = (numerator.bit_length() - denominator.bit_length()) * 1233 // 4096
    if exponent >= 0:
        denominator *= 10 ** exponent
    else:
        numerator *= 10 ** (-exponent)
    while numerator < denominator:
        numerator *= 10
    while numerator >= 10 * denominator:
        denominator *= 10
    return numerator // denominator


def count_digits(values, multiplier=1, divisor=1):
    counts = [0] * 9
    for value in values:
        digit = leading_digit(value * multiplier, divisor)
        counts[digit - 1] += 1
    return counts


def powers_of_two(size):
    for exponent in range(1, size + 1):
        yield 1 << exponent


def fibonacci(size):
    current, following = 1, 1
    for _ in range(size):
        yield current
        current, following = following, current + following


def log_upper_gamma_ratio(shape, x):
    """양의 정수 a에 대한 정규화 상측 불완전 감마 Q(a, x)의 로그."""
    # 부분적분을 반복하면 Q(a, x) = exp(-x) * Σ(x**k / k!), k=0..a-1.
    if x == 0:
        return 0.0
    log_x = math.log(x)
    log_terms = [k * log_x - math.lgamma(k + 1) for k in range(shape)]
    largest = max(log_terms)
    log_sum = largest + math.log(math.fsum(math.exp(t - largest) for t in log_terms))
    return min(0.0, log_sum - x)


def format_probability(log_p):
    if log_p >= math.log(0.001):
        return f"{math.exp(log_p):.6g}"
    # 확률이 float의 표현 범위보다 작아도 지수 표기로 출력한다.
    log10_p = log_p / math.log(10)
    exponent = math.floor(log10_p)
    mantissa = round(10 ** (log10_p - exponent), 5)
    if mantissa == 10:
        mantissa = 1
        exponent += 1
    return f"{mantissa:.5f}e{exponent}"


def mad_label(mad):
    if mad <= 0.006:
        return "close"
    if mad <= 0.012:
        return "acceptable"
    if mad <= 0.015:
        return "marginal"
    return "nonconformity"


def show_distribution(title, counts):
    size = sum(counts)
    proportions = [observed / size for observed in counts]
    expected = [size * probability for probability in BENFORD]
    chi_square = math.fsum(
        (observed - target) ** 2 / target
        for observed, target in zip(counts, expected)
    )
    log_p = log_upper_gamma_ratio(4, chi_square / 2)
    mad = math.fsum(
        abs(observed - target) for observed, target in zip(proportions, BENFORD)
    ) / len(BENFORD)

    print(f"\n{title} (표본 {size:,}개)")
    print(" 첫자리 |   관측 비율 |   이론 비율")
    for digit, observed, target in zip(DIGITS, proportions, BENFORD):
        print(f" {digit:>6} | {observed:>10.4%} | {target:>10.4%}")
    print(f" 카이제곱={chi_square:.4f}, 자유도=8")
    print(f" p값(상측)={format_probability(log_p)}")
    print(f" Nigrini MAD={mad:.6f} ({mad_label(mad)})")
    return log_p, mad


def record_result(title, passed):
    status = "SKIP" if passed is None else "PASS" if passed else "FAIL"
    results[title] = status
    print(f"판정  {status}")


results = {}

In [ ]:
# Chart language and style
import os
import platform
import sys
from pathlib import Path
from urllib.request import urlopen

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.text import Text

CHART_LANG = os.environ.get('CHART_LANG', 'ko')
CHART_COLORS = ('#2a78d6', '#eb6834', '#1baf7a')
CHART_TEXT = '#0b0b0b'
CHART_MUTED = '#52514e'
CHART_GRID = '#e4e3df'
CHART_BG = '#fcfcfb'
CHART_LIMIT = '#e34948'

in_colab = 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ
chart_font = matplotlib.rcParamsDefault['font.family']
if CHART_LANG == 'ko':
    # NanumGothic is licensed under the SIL OFL 1.1.
    nanum = Path('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    try:
        if not nanum.is_file():
            nanum = Path('fonts/NanumGothic-Regular.ttf')
            if not nanum.is_file():
                nanum.parent.mkdir(exist_ok=True)
                url = 'https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf'
                with urlopen(url, timeout=60) as response:
                    nanum.write_bytes(response.read())
        font_manager.fontManager.addfont(str(nanum))
        chart_font = font_manager.FontProperties(fname=str(nanum)).get_name()
    except Exception as exc:
        CHART_LANG = 'en'
        reason = ' '.join(str(exc).splitlines()) or type(exc).__name__
        print(f'차트 언어 en, 한글 글꼴 설정 실패, {reason}')
    else:
        print(f'차트 언어 ko, 글꼴 {chart_font}')
else:
    CHART_LANG = 'en'

matplotlib.rcParams.update({
    'figure.figsize': (8, 4.5),
    'figure.dpi': 200,
    'savefig.dpi': 200,
    'figure.facecolor': CHART_BG,
    'savefig.facecolor': CHART_BG,
    'axes.facecolor': CHART_BG,
    'axes.edgecolor': CHART_MUTED,
    'axes.labelcolor': CHART_TEXT,
    'axes.titlelocation': 'left',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.axisbelow': True,
    'axes.unicode_minus': False,
    'axes.prop_cycle': matplotlib.rcsetup.cycler(color=CHART_COLORS),
    'axes.grid': True,
    'axes.grid.axis': 'y',
    'grid.color': CHART_GRID,
    'grid.linestyle': '-',
    'grid.linewidth': 0.8,
    'text.color': CHART_TEXT,
    'xtick.color': CHART_MUTED,
    'ytick.color': CHART_MUTED,
    'lines.linewidth': 2,
    'lines.markersize': 6,
    'legend.frameon': False,
    'font.family': chart_font,
})

chart_location = 'Colab' if in_colab else platform.system()
CPU_RUNTIME = f'{chart_location} CPU ({platform.machine()}), Python {platform.python_version()}'


def finish_chart(fig, runtime=CPU_RUNTIME, top=1):
    for text in fig.findobj(Text):
        label = text.get_text()
        if '·' in label:
            text.set_text(label.replace(' ·', ',').replace('·', ','))
    runtime = runtime.replace(' ·', ',').replace('·', ',')
    fig.tight_layout(rect=(0, 0.065, 1, top))
    fig.text(0.02, 0.025, runtime, fontsize=8, color=CHART_MUTED)
    plt.show()

## 1. 이론 분포

첫자리 \(d\)의 확률은 \(\log_{10}(1+1/d)\)입니다. 첫 한 자리와 첫 두 자리의 이론 확률 합이 각각 1인지 확인합니다.

In [ ]:
def check_theory():
    first_sum = math.fsum(BENFORD)
    two_digit_sum = math.fsum(math.log10(1 + 1 / number) for number in range(10, 100))
    print(" 첫자리 | 이론 확률")
    for digit, probability in zip(DIGITS, BENFORD):
        print(f" {digit:>6} | {probability:.8f}")
    print(f"첫 한 자리 합    {first_sum:.15f}")
    print(f"첫 두 자리 합    {two_digit_sum:.15f}")
    tolerance = 1e-12
    print(f"판정 기준  각 합과 1의 차이가 {tolerance:g} 이하")
    return abs(first_sum - 1) <= tolerance and abs(two_digit_sum - 1) <= tolerance

record_result("이론 분포", check_theory())

## 2. 거듭제곱과 피보나치 수열

두 수열의 첫자리를 정수 연산으로 세고, 벤포드 분포와 비교합니다. MAD는 각 자리의 관측 비율과 이론 비율 사이의 절대 차이를 평균한 값입니다.

In [ ]:
def check_sequences(quick):
    power_size, fib_size = (1000, 500) if quick else (10_000, 5_000)
    _, power_mad = show_distribution("2의 거듭제곱", count_digits(powers_of_two(power_size)))
    _, fib_mad = show_distribution("피보나치 수열", count_digits(fibonacci(fib_size)))
    print("판정 기준  두 수열 모두 MAD 0.012 이하")
    return max(power_mad, fib_mad) <= 0.012

record_result("거듭제곱과 피보나치", check_sequences(QUICK))

## 3. 미국 카운티 인구 추정

미국 인구조사국 CSV를 내려받아 카운티 및 카운티 상당 지역의 인구 추정치를 사용합니다. 표본이 크면 작은 차이도 카이제곱 검정에서 유의하게 나올 수 있어 MAD를 함께 봅니다.

In [ ]:
import csv
import http.client
import io
import queue
import threading
import urllib.request

# 출처: 미국 인구조사국 Population Estimates Program의 2020~2023년 카운티 인구 추정 CSV.
# 이용 조건: 미국 연방 정부 저작물, 퍼블릭 도메인.
CENSUS_URL = (
    "https://www2.census.gov/programs-surveys/popest/datasets/"
    "2020-2023/counties/totals/co-est2023-alldata.csv"
)
DOWNLOAD_LIMIT = 25


def download_county_populations():
    result = queue.Queue(maxsize=1)

    def fetch():
        try:
            with urllib.request.urlopen(CENSUS_URL, timeout=10) as response:
                with io.TextIOWrapper(response, encoding="latin-1", newline="") as source:
                    reader = csv.DictReader(source, strict=True)
                    if not {"SUMLEV", "POPESTIMATE2023"}.issubset(reader.fieldnames or ()):
                        result.put((None, "CSV에 필요한 열이 없습니다."))
                        return
                    populations = [
                        int(row["POPESTIMATE2023"])
                        for row in reader if row["SUMLEV"] == "050"
                    ]
            if not populations or min(populations) <= 0:
                result.put((None, "양의 카운티 인구를 확인할 수 없습니다."))
                return
            result.put((populations, None))
        except (OSError, ValueError, TypeError, csv.Error, http.client.HTTPException) as error:
            result.put((None, f"자료를 읽지 못했습니다 ({type(error).__name__})."))

    # 소켓의 개별 대기 시간과 별도로 실험 전체 다운로드 대기를 제한한다.
    threading.Thread(target=fetch, daemon=True).start()
    try:
        return result.get(timeout=DOWNLOAD_LIMIT)
    except queue.Empty:
        return None, f"다운로드 제한 시간({DOWNLOAD_LIMIT}초)을 넘었습니다."

populations, error = download_county_populations()
baseline = None
if populations is None:
    print(error)
    record_result("미국 카운티 인구 추정", None)
else:
    baseline = count_digits(populations)
    show_distribution("2023년 카운티 추정 인구", baseline)
    record_result("미국 카운티 인구 추정", True)

In [ ]:
from matplotlib.ticker import PercentFormatter

L = {
    'ko': {
        'title': '미국 카운티 인구의 첫자리 분포', 'x': '첫자리 숫자', 'y': '비율',
        'observed': '관측', 'theory': '벤포드 이론', 'source': '미국 인구조사국 2023년 인구 추정',
        'skip': '카운티 자료가 없어 첫자리 그래프를 건너뜁니다.',
    },
    'en': {
        'title': 'Leading digits of U.S. county populations', 'x': 'Leading digit', 'y': 'Share',
        'observed': 'Observed', 'theory': "Benford's law", 'source': 'U.S. Census Bureau 2023 population estimates',
        'skip': 'Skipping the leading-digit chart because county data is unavailable.',
    },
}
labels = L[CHART_LANG]
if baseline is None:
    print(labels['skip'])
else:
    county_size = sum(baseline)
    fig, ax = plt.subplots()
    ax.bar(DIGITS, [count / county_size for count in baseline], width=0.65,
           color=CHART_COLORS[0], label=labels['observed'])
    ax.plot(DIGITS, BENFORD, color=CHART_COLORS[1], marker='o', label=labels['theory'])
    ax.set(title=labels['title'], xlabel=labels['x'], ylabel=labels['y'])
    ax.set_xticks(DIGITS)
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.legend(loc='upper right')
    finish_chart(fig, f"{CPU_RUNTIME} · {labels['source']} · n={county_size:,}")

## 4. 배율 변환

같은 인구 자료에 서로 다른 양의 배율을 적용합니다. 변환 전후 첫자리 분포가 얼마나 달라지는지 자리별 최대 변화와 총변동 거리로 비교합니다.

In [ ]:
def check_scaling(populations, baseline):
    if populations is None:
        print("실험 3의 카운티 자료가 필요합니다.")
        return None

    size = len(populations)
    max_changes = []
    for label, multiplier in (("1.37", 137), ("0.73", 73), ("3.28", 328)):
        scaled = count_digits(populations, multiplier, 100)
        show_distribution(f"카운티 인구 × {label}", scaled)
        changes = [abs(after - before) / size for after, before in zip(scaled, baseline)]
        max_changes.append(max(changes))
        print(f" 원본 대비 최대 자리별 변화={100 * max_changes[-1]:.4f}%p")
        print(f" 원본 대비 총변동 거리={math.fsum(changes) / 2:.6f}")

    print("판정 기준  자리별 변화가 모두 3%p 이하")
    return max(max_changes) <= 0.03

record_result("배율 변환", check_scaling(populations, baseline))

## 5. 벤포드 분포와 다른 표본

균등 정수 난수, 좁은 범위의 키 모형, 첫자리를 균등하게 뽑은 합성 표본을 비교합니다. 시드를 고정해 같은 표본을 다시 만들 수 있습니다.

In [ ]:
def check_counterexamples(quick):
    size = 1000 if quick else 10_000
    rng = random.Random(SEED)
    uniform = (rng.randint(1, 9999) for _ in range(size))
    uniform_counts = count_digits(uniform)
    uniform_result = show_distribution("균등 난수 1~9999", uniform_counts)

    heights = []
    while len(heights) < size:
        height = round(rng.gauss(170, 10))
        if height > 0:
            heights.append(height)
    height_result = show_distribution("키 모형 (평균 170, 표준편차 10)", count_digits(heights))

    synthetic_counts = [0] * 9
    for _ in range(size):
        synthetic_counts[rng.randrange(9)] += 1
    synthetic_result = show_distribution("균등 첫자리 합성", synthetic_counts)

    print("판정 기준  세 표본 모두 p < 0.05, MAD > 0.015")
    passed = all(
        log_p < math.log(0.05) and mad > 0.015
        for log_p, mad in (uniform_result, height_result, synthetic_result)
    )
    return passed, uniform_counts

counterexamples_passed, uniform_counts = check_counterexamples(QUICK)
record_result("벤포드를 따르지 않는 표본", counterexamples_passed)

In [ ]:
from matplotlib.ticker import PercentFormatter

L = {
    'ko': {
        'title': '균등 정수 난수의 첫자리 분포', 'x': '첫자리 숫자', 'y': '비율',
        'observed': '관측', 'theory': '벤포드 이론', 'source': '1~9,999에서 균등 추출',
    },
    'en': {
        'title': 'Leading digits of uniform random integers', 'x': 'Leading digit', 'y': 'Share',
        'observed': 'Observed', 'theory': "Benford's law", 'source': 'Uniform draws from 1 to 9,999',
    },
}
labels = L[CHART_LANG]
uniform_size = sum(uniform_counts)
fig, ax = plt.subplots()
ax.bar(DIGITS, [count / uniform_size for count in uniform_counts], width=0.65,
       color=CHART_COLORS[0], label=labels['observed'])
ax.plot(DIGITS, BENFORD, color=CHART_COLORS[1], marker='o', linestyle='--', label=labels['theory'])
ax.set(title=labels['title'], xlabel=labels['x'], ylabel=labels['y'])
ax.set_xticks(DIGITS)
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.legend(loc='upper right')
finish_chart(fig, f"{CPU_RUNTIME} · {labels['source']} · n={uniform_size:,}")

## 판정 요약

자료를 내려받지 못하면 인구와 배율 변환 실험은 `SKIP`으로 표시됩니다.

In [ ]:
print(" 번호 | 판정 | 실험")
for number, (title, status) in enumerate(results.items(), start=1):
    print(f" {number:>4} | {status:<4} | {title}")